# Late Interaction using ColBert
ColBERT is a late interaction technique to improve search quality.
<br>
In **Late Interaction** using ColBERT :
* embeddings are created for every token in the search query and the document.
* for each query token we find a document token with maximum cosine similarity.
* these maximum similarity (`MaxSim`) scores are aggregated to produce the final document relevance score.

> **NOTE**<br> An honest ColBERT implementation requires one vector per token of each document and MaxSim scoring serverside.<br>
However, turbopuffer doesn't support multi-vector documents today.

Since turbopuffer does not natively support multi-vector documents, a practical ColBERT implementation today is a two-stage approximation:
* dense ANN retrieval followed by 
* client-side MaxSim re-ranking on the top-k candidates.

Key implementation desciions:
* At index time, ColBERT is run on Every document and token vectors vectors are stored as JSON attribute on each document row.
* At query time:    
    * turbopuffer ANN finds top-100 dense vectors fast
    * turbopuffer trturns token_vectors attribute for those 100 docs in the same response (no additional round trip)
    * client computes MaxSim on the pre-computed token matrices.
Advantages of this solution:
* Pre-computed tokens once at index-time, makes every query fast. 
* Single roundtrip to turbopuffer.
* Token vectors trabel with the document, no lookups or joins.
* Storing JSOB blobs is cheaper in turbopuffer's storage model.


import turbopuffer
from dotenv import load_dotenv
import os

# Load variables from .env
load_dotenv()

# Access variables
TURBOPUFFER_API_KEY = os.getenv("TURBOPUFFER_API_KEY")